<a href="https://colab.research.google.com/github/kartikigaikwad/Amazon-Clone/blob/main/kartiki_App_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U openai==1.55.3 langchain langchain-openai langchain-core pinecone gradio arxiv tqdm

import arxiv
import openai
import pinecone
import os
from tqdm import tqdm
from google.colab import userdata
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import AzureOpenAIEmbeddings
from pinecone import Pinecone, ServerlessSpec, PineconeApiException
import gradio as gr

# Load secrets from Colab userdata
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = userdata.get(f"{var}")

for var in ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT", "USER_AGENT",
            "PINECONE_API_KEY", "PINECONE_INDEX", "OPENAI_API_VERSION"]:
    _set_env(var)

openai.api_key = os.getenv("AZURE_OPENAI_API_KEY")
openai.api_base = os.getenv("AZURE_OPENAI_ENDPOINT")
openai.api_type = "azure"
openai.api_version = os.getenv("OPENAI_API_VERSION")

pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_index_name = os.getenv("PINECONE_INDEX")

pinecone_client = Pinecone(api_key=pinecone_api_key)

# Create index if not present
try:
    existing_indexes = [i['name'] for i in pinecone_client.list_indexes()]
    if pinecone_index_name not in existing_indexes:
        print(f"Creating new Pinecone index: {pinecone_index_name}")
        pinecone_client.create_index(
            name=pinecone_index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
    else:
        print(f"Pinecone index '{pinecone_index_name}' already exists.")
except PineconeApiException as e:
    if e.status == 409 and "ALREADY_EXISTS" in e.body:
        print(f"Pinecone index '{pinecone_index_name}' already exists.")
    else:
        raise

index = pinecone_client.Index(pinecone_index_name)
print(f" Connected to Pinecone index: {pinecone_index_name}")


  Using cached langchain-1.1.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_openai-1.1.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_core-1.1.0-py3-none-any.whl.metadata (3.6 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.0.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_openai-1.0.2-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_openai-1.0.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_openai-1.0.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_openai-0.3.35-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_openai-0.3.34-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_openai-0.3.33-py3-none-any.whl.metadata (2.4 kB)
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other req

In [ ]:
"""
Agentic Research Assistant — Unified app (Part-2)
Updated script: auto-detect topic, check namespace, auto-create & ingest if missing,
and automatically answer queries from existing or newly-created namespaces.
"""

import os
import uuid
import json
import time
from typing import List, Tuple, Dict

import arxiv
import gradio as gr
from tqdm import tqdm

# LangChain splitter & Azure embeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import AzureOpenAIEmbeddings

# Pinecone client from your earlier code
from pinecone import Pinecone, ServerlessSpec, PineconeApiException

# Azure Chat client
from openai import AzureOpenAI

# -----------------------------
# Environment / Client Setup
# -----------------------------

for var in ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT", "OPENAI_API_VERSION",
            "PINECONE_API_KEY", "PINECONE_INDEX", "USER_AGENT"]:
    if not os.getenv(var):
        print(f"Warning: environment variable {var} not set")

openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
openai_api_base = os.getenv("AZURE_OPENAI_ENDPOINT")
openai_api_version = os.getenv("OPENAI_API_VERSION")

pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_index_name = os.getenv("PINECONE_INDEX") or "research-index"

# Initialize Pinecone client
pinecone_client = Pinecone(api_key=pinecone_api_key)

try:
    existing_indexes = [i['name'] for i in pinecone_client.list_indexes()]
    if pinecone_index_name not in existing_indexes:
        print(f"Creating Pinecone index: {pinecone_index_name}")
        pinecone_client.create_index(
            name=pinecone_index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
    else:
        print(f"Pinecone index '{pinecone_index_name}' exists")
except PineconeApiException as e:
    try:
        if e.status == 409 and "ALREADY_EXISTS" in e.body:
            print(f"Pinecone index '{pinecone_index_name}' already exists (409)")
    except Exception:
        raise

index = pinecone_client.Index(pinecone_index_name)
print(f"Connected to Pinecone index: {pinecone_index_name}")

# Azure chat client wrapper
azure_client = AzureOpenAI(
    api_key=openai_api_key,
    azure_endpoint=openai_api_base,
    api_version=openai_api_version
)
# Default model name (replace if needed to match your Azure deployment)
AZURE_CHAT_MODEL = os.getenv("AZURE_CHAT_MODEL", "gpt-4.1")

# Embedding model default
DEFAULT_EMBED_MODEL = "text-embedding-3-small"

# -----------------------------
# Helper functions (search, chunk, embed, upsert, query)
# -----------------------------

def search_arxiv(query: str, max_results: int = 5) -> List[dict]:
    s = arxiv.Search(query=query, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)
    results = []
    for r in s.results():
        results.append({
            "id": r.get_short_id(),
            "title": r.title,
            "authors": [a.name for a in r.authors],
            "summary": r.summary,
            "pdf_url": r.pdf_url,
            "published": r.published.isoformat() if r.published else None
        })
    return results


def chunk_texts(texts: List[str], chunk_size: int = 1000, overlap: int = 200) -> List[str]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    joined = "\n\n".join(texts)
    return splitter.split_text(joined)


def get_embeddings_client(deployment: str = DEFAULT_EMBED_MODEL):
    return AzureOpenAIEmbeddings(deployment=deployment)


def embed_texts(emb_client, texts: List[str], batch_size: int = 8) -> List[List[float]]:
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch_embs = emb_client.embed_documents(batch)
        embeddings.extend(batch_embs)
    return embeddings


def upsert_to_pinecone(index, embeddings: List[List[float]], texts: List[str], metadatas: List[dict], namespace: str = "research"):
    vectors = []
    for i, emb in enumerate(embeddings):
        vid = metadatas[i].get("vector_id") if metadatas[i].get("vector_id") else str(uuid.uuid4())
        vectors.append((vid, emb, metadatas[i]))
    index.upsert(vectors=vectors, namespace=namespace)


# Ingest pipeline (similar to your second app)
def ingest_arxiv_topic_to_pinecone(topic: str, max_results: int = 10, chunk_size: int = 1000, overlap: int = 200, namespace: str = "research", embedding_deployment: str = DEFAULT_EMBED_MODEL):
    papers = search_arxiv(topic, max_results=max_results)
    if not papers:
        return [], "No papers found"

    texts = []
    metadatas = []
    for p in papers:
        paper_text = f"Title: {p['title']}\nAuthors: {', '.join(p['authors'])}\nPublished: {p['published']}\nURL: {p['pdf_url']}\n\nAbstract:\n{p['summary']}"
        texts.append(paper_text)
        metadatas.append({
            "paper_id": p["id"],
            "title": p["title"],
            "authors": p["authors"],
            "pdf_url": p["pdf_url"],
            "published": p["published"]
        })

    # chunk per paper to keep traceability
    all_chunks = []
    all_chunk_meta = []
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    for i, text in enumerate(texts):
        chunks = splitter.split_text(text)
        for ci, c in enumerate(chunks):
            all_chunks.append(c)
            meta = metadatas[i].copy()
            meta.update({"chunk_index": ci, "text": c, "vector_id": f"{metadatas[i]['paper_id']}__chunk{ci}"})
            all_chunk_meta.append(meta)

    emb_client = get_embeddings_client(deployment=embedding_deployment)
    chunk_embeddings = embed_texts(emb_client, all_chunks, batch_size=8)
    upsert_to_pinecone(index, chunk_embeddings, all_chunks, all_chunk_meta, namespace=namespace)

    return papers, f"Upserted {len(chunk_embeddings)} vectors into namespace '{namespace}'"


# Retrieval pipeline
def retrieve_chunks_with_metadata(query: str, top_k: int = 5, namespace: str = "research", embedding_deployment: str = DEFAULT_EMBED_MODEL) -> Tuple[List[str], List[dict]]:
    emb_client = get_embeddings_client(deployment=embedding_deployment)
    query_emb = emb_client.embed_query(query)
    res = index.query(vector=query_emb, top_k=top_k, include_metadata=True, include_values=False, namespace=namespace)
    matches = res.get("matches", [])
    if not matches:
        return [], []

    retrieved_chunks = []
    citations_map = {}
    for m in matches:
        meta = m.get("metadata", {})
        chunk_text = meta.get("text", "")
        retrieved_chunks.append(chunk_text)
        paper_id = meta.get("paper_id", f"unknown_{m.get('id')}")
        score = m.get("score")
        if paper_id not in citations_map:
            citations_map[paper_id] = {
                "paper_id": paper_id,
                "title": meta.get("title"),
                "authors": meta.get("authors"),
                "pdf_url": meta.get("pdf_url"),
                "published": meta.get("published"),
                "top_score": score,
                "references": [{"chunk_index": meta.get("chunk_index"), "score": score}]
            }
        else:
            citations_map[paper_id]["references"].append({"chunk_index": meta.get("chunk_index"), "score": score})
            if score and (citations_map[paper_id]["top_score"] is None or score > citations_map[paper_id]["top_score"]):
                citations_map[paper_id]["top_score"] = score

    citations = sorted(citations_map.values(), key=lambda x: (x.get("top_score") is not None, x.get("top_score")), reverse=True)
    return retrieved_chunks, citations


# LLM answer generation (grounded)
def generate_answer_and_citations(query: str, retrieved_chunks: List[str], citations: List[dict]) -> str:
    context = "\n\n---\n\n".join(retrieved_chunks)
    citations_text_lines = []
    for idx, c in enumerate(citations, start=1):
        title = c.get("title") or "Untitled"
        authors = ", ".join(c.get("authors") or [])
        pdf = c.get("pdf_url") or ""
        published = c.get("published") or ""
        citations_text_lines.append(f"[Ref {idx}] {title} — {authors}. {published}. {pdf}")
    citations_text = "\n".join(citations_text_lines)

    system_prompt = f"""
You are an expert research assistant. Use ONLY the provided context to answer the user's question.
- When you use information from the context, append a citation token like [Ref 1], [Ref 2], etc.
- Do NOT hallucinate. If information is missing, reply: "Insufficient research context."

Context:
{context}

Citations:
{citations_text}
"""

    response = azure_client.chat.completions.create(
        model=AZURE_CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0,
        max_tokens=800,
    )

    answer = response.choices[0].message.content

    md_citations = []
    for idx, c in enumerate(citations, start=1):
        title = c.get("title") or "Untitled"
        authors = ", ".join(c.get("authors") or [])
        pdf = c.get("pdf_url") or ""
        published = c.get("published") or ""
        md_line = f"- **[Ref {idx}]** [{title}]({pdf}) — {authors} — {published}"
        md_citations.append(md_line)

    final_output = f"### Answer\n\n{answer}\n\n---\n\n### Cited Papers\n\n" + "\n".join(md_citations)
    return final_output


# -----------------------------
# Agent logic: decide intent & manage state
# -----------------------------

INDEX_KEYWORDS = ["index", "ingest", "ingesting", "ingestion", "ingest arxiv", "indexing", "add papers", "fetch papers", "ingest topic", "ingest arxiv", "ingest"]
NEW_TOPIC_KEYWORDS = ["new topic", "start new", "reset", "start over", "new research", "clear namespace", "change topic"]

# State management for namespaces
state: Dict = {"namespaces": {}, "current_namespace": None}


def detect_intent(user_text: str) -> str:
    text = user_text.lower()
    if any(k in text for k in NEW_TOPIC_KEYWORDS):
        return "new_topic"
    if any(k in text for k in INDEX_KEYWORDS):
        return "index"
    if text.strip().startswith("topic:") or text.strip().startswith("namespace:"):
        return "index"
    if text.strip().endswith("?") or text.strip().split()[0] in ["what", "how", "why", "compare", "explain", "summarize"]:
        return "query"
    return "query"


def ensure_namespace(ns: str, topic: str = None):
    if ns not in state["namespaces"]:
        state["namespaces"][ns] = {"status": "idle", "last_indexed": None, "topic": topic}
    else:
        if topic:
            state["namespaces"][ns]["topic"] = topic


# -----------------------------
# NEW helpers: topic guessing & pinecone namespace check
# -----------------------------

def guess_topic_from_query(text: str) -> str:
    """
    Simple heuristic topic extractor. Keeps 1-3 keywords as namespace.
    Replace with a GPT extraction if you want more accuracy.
    """
    if not text:
        return "general-topic"
    words = text.lower().replace("?", " ").replace(".", " ").split()
    stopwords = {"what","how","why","explain","compare","summarize","is","the","of","in","on","for","and","to","a","an"}
    keywords = [w for w in words if w not in stopwords and len(w) > 3]
    # fallback
    if not keywords:
        return "general-topic"
    return "-".join(keywords[:3])


def namespace_exists_in_pinecone(ns: str) -> bool:
    """
    Uses index.describe_index_stats() to check whether namespace exists.
    """
    try:
        stats = index.describe_index_stats()
        namespaces = stats.get("namespaces", {}) or {}
        return ns in namespaces
    except Exception:
        # if API fails, fall back to state tracking
        return ns in state["namespaces"] and state["namespaces"][ns].get("status") == "ready"


# -----------------------------
# UPDATED agent_handle: auto-detect + auto-index + auto-answer
# -----------------------------
def agent_handle(user_message: str, topic_input: str, top_k: int, max_papers: int, chunk_size: int, chunk_overlap: int, embed_model: str):
    logs = []

    # New-topic explicit intent
    intent = detect_intent(user_message if user_message else topic_input or "")
    if intent == "new_topic":
        new_ns = f"ns_{int(time.time())}"
        parsed_topic = topic_input if topic_input else user_message
        ensure_namespace(new_ns, parsed_topic)
        state['current_namespace'] = new_ns
        logs.append(f"🔁 Started new research namespace '{new_ns}' for topic: {parsed_topic}")
        logs.append("You can index papers by writing 'index' + the topic, or put the topic in the 'Topic' box and then ask the assistant to index it.")
        return "\n".join(logs), "", json.dumps(state)

    # Determine topic & namespace
    topic = topic_input.strip() if topic_input else guess_topic_from_query(user_message)
    ns = topic.replace(" ", "_").lower()
    logs.append(f"🔍 Extracted topic: **{topic}**")
    logs.append(f"🔧 Using namespace: **{ns}**")
    ensure_namespace(ns, topic)
    state['current_namespace'] = ns

    # If user explicitly asked to index
    if intent == "index":
        if not topic:
            logs.append("❗ To index, please provide a topic in the Topic box or write 'index <your topic>'.")
            return "\n".join(logs), "", json.dumps(state)

        logs.append(f"🚀 Agent detected indexing intent for namespace '{ns}' and topic: {topic}")
        state['namespaces'][ns]['status'] = 'indexing'
        state['namespaces'][ns]['topic'] = topic
        state['namespaces'][ns]['last_indexed'] = None

        papers, status = ingest_arxiv_topic_to_pinecone(topic=topic, max_results=max_papers, chunk_size=int(chunk_size), overlap=int(chunk_overlap), namespace=ns, embedding_deployment=embed_model)
        logs.append(status)
        if papers:
            state['namespaces'][ns]['status'] = 'ready'
            state['namespaces'][ns]['last_indexed'] = time.time()
            logs.append(f"✅ Indexing finished for namespace '{ns}'. You can now ask research questions about this topic.")
            paper_lines = [f"{i+1}. {p['title']} — {', '.join(p['authors'])}" for i, p in enumerate(papers)]
            results_html = "<br>".join(paper_lines)
            return "\n".join(logs), results_html, json.dumps(state)
        else:
            state['namespaces'][ns]['status'] = 'idle'
            logs.append("⚠ No papers were indexed.")
            return "\n".join(logs), "", json.dumps(state)

    # QUERY FLOW: auto-check namespace, auto-index if missing
    if namespace_exists_in_pinecone(ns) or (ns in state["namespaces"] and state["namespaces"][ns].get("status") == "ready"):
        logs.append("🟢 Topic already indexed — retrieving relevant answers...")
        retrieved_chunks, citations = retrieve_chunks_with_metadata(user_message, top_k=top_k, namespace=ns, embedding_deployment=embed_model)
        if not retrieved_chunks:
            logs.append("⚠ No relevant context found in the existing namespace. Consider indexing more papers or increasing top_k.")
            return "\n".join(logs), "", json.dumps(state)
        answer_md = generate_answer_and_citations(user_message, retrieved_chunks, citations)
        return "\n".join(logs + [f"✅ Answer generated using namespace '{ns}'"]), answer_md, json.dumps(state)

    # If not indexed → auto-ingest then answer
    logs.append("🟡 Topic not indexed — creating namespace & indexing now...")
    ensure_namespace(ns, topic)
    state["current_namespace"] = ns
    state["namespaces"][ns]["status"] = "indexing"

    papers, status = ingest_arxiv_topic_to_pinecone(topic=topic, max_results=max_papers, chunk_size=int(chunk_size), overlap=int(chunk_overlap), namespace=ns, embedding_deployment=embed_model)
    logs.append(status)

    if not papers:
        state["namespaces"][ns]["status"] = "idle"
        logs.append("❌ No papers found to index. Try a broader topic or different keywords.")
        return "\n".join(logs), "", json.dumps(state)

    state["namespaces"][ns]["status"] = "ready"
    state["namespaces"][ns]["last_indexed"] = time.time()
    logs.append("🔵 Indexing complete — generating answer now...")

    retrieved_chunks, citations = retrieve_chunks_with_metadata(user_message, top_k=top_k, namespace=ns, embedding_deployment=embed_model)
    if not retrieved_chunks:
        logs.append("⚠ Indexing done, but no relevant context found to answer your query.")
        return "\n".join(logs), "", json.dumps(state)

    answer_md = generate_answer_and_citations(user_message, retrieved_chunks, citations)
    logs.append("✅ Answer retrieved from freshly indexed data.")
    return "\n".join(logs), answer_md, json.dumps(state)


# -----------------------------
# Gradio UI
# -----------------------------

def build_ui():
    with gr.Blocks(title="Agentic Research Assistant — Unified") as app:
        gr.Markdown("# 🧠 Agentic Research Assistant — Unified (Part-2)")
        gr.Markdown("This single interface detects whether you want to index (ingest) papers or ask research questions and manages state for namespaces/topics.")

        with gr.Row():
            with gr.Column(scale=2):
                topic_input = gr.Textbox(label="Topic (optional)", placeholder="e.g., graph neural networks")
                user_message = gr.Textbox(label="Message / Query", placeholder="Ask a question or type 'index <topic>' to ingest papers")
                embed_model_input = gr.Dropdown(choices=[DEFAULT_EMBED_MODEL, "text-embedding-3-large"], value=DEFAULT_EMBED_MODEL, label="Embedding model")
                max_papers_input = gr.Slider(1, 50, value=10, step=1, label="Max papers to fetch when indexing")
                chunk_size_input = gr.Number(label="Chunk size", value=1000)
                chunk_overlap_input = gr.Number(label="Chunk overlap", value=200)
                top_k_input = gr.Slider(1, 12, value=5, step=1, label="Top K results for retrieval")
                submit_btn = gr.Button("Send")
                clear_btn = gr.Button("Start new topic / Reset")

            with gr.Column(scale=3):
                logs_out = gr.Markdown(label="Agent Logs / Notifications")
                results_out = gr.Markdown(label="Results (Answer or Index Summary)")
                state_out = gr.JSON(value=state, label="Agent State (namespaces)")

        def on_submit(user_message_, topic_, top_k_, max_papers_, chunk_size_, chunk_overlap_, embed_model_):
            logs, results, new_state = agent_handle(user_message_, topic_, top_k_, max_papers_, chunk_size_, chunk_overlap_, embed_model_)
            try:
                parsed = json.loads(new_state)
            except:
                parsed = state
            return logs, results, parsed

        def on_clear(_):
            new_ns = f"ns_{int(time.time())}"
            ensure_namespace(new_ns, topic=None)
            state['current_namespace'] = new_ns
            state['namespaces'][new_ns]['status'] = 'idle'
            return f"🔁 New namespace created: {new_ns}", "", state

        submit_btn.click(on_submit, inputs=[user_message, topic_input, top_k_input, max_papers_input, chunk_size_input, chunk_overlap_input, embed_model_input], outputs=[logs_out, results_out, state_out])
        clear_btn.click(on_clear, inputs=[topic_input], outputs=[logs_out, results_out, state_out])

    return app


if __name__ == '__main__':
    ui = build_ui()
    ui.launch(debug=True)


Pinecone index 'my-pro' exists
Connected to Pinecone index: my-pro
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a0aaab29aeb9da3d1a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipython-input-1716237333.py:87: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for r in s.results():
